# 26. 고장 해석 (Model Interpretation)

목적: 학습된 앙상블 모델에 대해 SHAP 기반의 다층적 해석을 수행합니다.

| 섹션 | 내용 |
| --- | --- |
| **§9.1 전역적 해석 (Global)** | SHAP Summary Plot — 전체 데이터 기준 주요 고장 기여 피처 분석 |
| **§9.2 국소적 해석 (Local)** | SHAP Waterfall Plot — 특정 개체(시리얼)의 고장 예측 근거 설명 |
| **§9.3 시간 기반 해석 (Temporal)** | 고장 전 N일 창에서의 SHAP 기여도 변화 trajectory 추적 |

> ⚠️ 사전 조건: `07_threshold_tuning.ipynb` 및 `08_final_evaluation.ipynb` 실행 완료 후 진행하세요.

## 0. 환경 준비

In [ ]:
import sys, os, json, joblib
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

sys.path.insert(0, os.path.abspath("..\\"))

import config.interp_config as cfg
import config.train_config as tcfg

warnings.filterwarnings("ignore")

# 한글 폰트 설정
for _font in ["Malgun Gothic", "NanumGothic", "AppleGothic", "DejaVu Sans"]:
    if any(_font.lower() in f.name.lower() for f in fm.fontManager.ttflist):
        plt.rcParams["font.family"] = _font
        break
plt.rcParams["axes.unicode_minus"] = False

print("환경 준비 완료")

## 1. 모델 및 데이터 로드

In [ ]:
from pathlib import Path

SAVE_DIR = Path(tcfg.MODEL_SAVE_DIR)

# ── 피처 목록 로드 ─────────────────────────────────────────
with open(SAVE_DIR / "feature_cols.json", encoding="utf-8") as f:
    FEATURE_COLS = json.load(f)

# ── 최적 임계값 로드 ───────────────────────────────────────
with open(SAVE_DIR / "best_threshold.json", encoding="utf-8") as f:
    THRESHOLD = json.load(f)["threshold"]

# ── 앙상블 모델 로드 ───────────────────────────────────────
models = [joblib.load(p) for p in sorted(SAVE_DIR.glob("subset_*.pkl"))]

# ── 테스트 데이터 로드 ─────────────────────────────────────
df_test = pd.read_parquet(cfg.TEST_PATH)
df_test[cfg.DATE_COL] = pd.to_datetime(df_test[cfg.DATE_COL])

# ── 앙상블 Soft-voting 예측 확률 계산 (1회만 실행) ──────────
print(f"앙상블 추론 중... (모델 {len(models)}개 × 데이터 {len(df_test):,}행)")
y_prob = np.mean(
    [m.predict_proba(df_test[FEATURE_COLS])[:, 1] for m in models], axis=0
)
df_test["_prob"]  = y_prob
df_test["_alarm"] = (y_prob >= THRESHOLD).astype(int)

print(f"✅  로드 완료: 모델 {len(models)}개 | 피처 {len(FEATURE_COLS)}개 | 임계값 {THRESHOLD:.4f}")
print(f"    테스트 데이터: {len(df_test):,}행 | 고장행: {df_test[cfg.TARGET_COL].sum():,}")

---
## §9.1 전역적 해석 (Global Interpretability)

SHAP Summary Plot을 통해 전체 테스트 데이터에서 어떤 피처가 고장 예측에 얼마나, 어떤 방향으로 기여하는지 분석합니다.

- 앙상블 모델 전체의 평균 SHAP 값을 사용
- **x축:** 개별 예측에 대한 SHAP 값 (양수 = 고장 확률 증가에 기여)
- **색상:** 해당 피처의 원본 값 수준 (붉을수록 높은 값)

In [ ]:
import shap

# SHAP 계산용 샘플 추출 (전체 사용 시 매우 느림)
sample_size = min(cfg.SHAP_SAMPLE_SIZE, len(df_test))
X_sample = (
    df_test[FEATURE_COLS]
    .sample(sample_size, random_state=cfg.SHAP_SEED)
    .reset_index(drop=True)
)

print(f"SHAP 계산 중... (샘플={sample_size:,}, 모델={len(models)}개)")
print("  ※ 모델 수 × 샘플 수에 비례하여 수 분 소요될 수 있습니다.")

sv_list = []
for i, model in enumerate(models):
    explainer = shap.TreeExplainer(model)
    sv = explainer.shap_values(X_sample)
    if isinstance(sv, list):
        sv = sv[1]  # 이진 분류: 양성 클래스(고장) SHAP
    sv_list.append(sv)
    print(f"  모델 {i+1}/{len(models)} 완료", end="\r")

mean_shap = np.mean(sv_list, axis=0)
print(f"\n✅  SHAP 계산 완료!")

In [ ]:
# §9.1 Summary Plot (Beeswarm)
print("[§9.1] SHAP Summary Plot (전역 해석)")
shap.summary_plot(
    mean_shap,
    X_sample,
    feature_names=FEATURE_COLS,
    max_display=cfg.SHAP_MAX_DISPLAY,
    show=True,
    plot_size=(10, 8),
)

In [ ]:
# §9.1 Bar Plot — 절댓값 평균 기준 피처 중요도
print("[§9.1] SHAP Bar Plot (Mean |SHAP| 기준 피처 중요도)")
shap.summary_plot(
    mean_shap,
    X_sample,
    feature_names=FEATURE_COLS,
    max_display=cfg.SHAP_MAX_DISPLAY,
    plot_type="bar",
    show=True,
    plot_size=(10, 8),
)

# 중요도 순위 테이블 출력
mean_abs_shap = np.abs(mean_shap).mean(axis=0)
shap_rank_df = (
    pd.DataFrame({"feature": FEATURE_COLS, "mean_abs_shap": mean_abs_shap})
    .sort_values("mean_abs_shap", ascending=False)
    .reset_index(drop=True)
)
shap_rank_df.index += 1
print("\n📊 피처 중요도 순위 (Mean |SHAP|):")
print(shap_rank_df.to_markdown())

---
## §9.2 국소적 해석 (Local Interpretability)

특정 개체(시리얼 번호)에 대해 SHAP Waterfall Plot을 통해 고장 예측의 근거를 설명합니다.

- 고장이 났으나 탐지하지 못한 **미탐 개체** 또는 **탐지 성공 개체**를 선택하여 분석
- 어떤 피처가 어느 방향으로 고장 확률을 끌어올리거나 낮췄는지 확인 가능

In [ ]:
# 분석 대상 시리얼 선택
TARGET_SERIAL = cfg.TARGET_SERIAL

if TARGET_SERIAL is None:
    # 미탐 개체(고장났으나 알람 미발생) 중 자동 선택
    fail_serials = df_test[df_test[cfg.TARGET_COL] == 1][cfg.SERIAL_COL].unique()
    alarm_serials = df_test[df_test["_alarm"] == 1][cfg.SERIAL_COL].unique()
    miss_serials = [s for s in fail_serials if s not in alarm_serials]

    if miss_serials:
        TARGET_SERIAL = miss_serials[0]
        print(f"[자동 선택] 미탐 고장 개체: {TARGET_SERIAL}")
        print(f"  → 미탐 개체 총 {len(miss_serials)}개 중 첫 번째 선택")
        print(f"  ※ config/interp_config.py 의 TARGET_SERIAL 로 직접 지정 가능합니다.")
    else:
        # 미탐 없으면 탐지 성공한 고장 개체 선택
        hit_serials = [s for s in fail_serials if s in alarm_serials]
        TARGET_SERIAL = hit_serials[0] if hit_serials else fail_serials[0]
        print(f"[자동 선택] 탐지 성공 고장 개체: {TARGET_SERIAL} (미탐 개체 없음)")
else:
    print(f"[수동 지정] 분석 대상: {TARGET_SERIAL}")

# 해당 시리얼의 데이터 추출 (시간순 정렬)
df_entity = df_test[df_test[cfg.SERIAL_COL] == TARGET_SERIAL].sort_values(cfg.DATE_COL)
print(f"\n  데이터: {len(df_entity)}행 | 고장행: {df_entity[cfg.TARGET_COL].sum()} | "
      f"알람 발생: {df_entity['_alarm'].sum()}")
print(f"  관측 기간: {df_entity[cfg.DATE_COL].min().date()} ~ {df_entity[cfg.DATE_COL].max().date()}")

In [ ]:
# §9.2 Waterfall Plot — 고장 확률이 가장 높은 시점
X_entity = df_entity[FEATURE_COLS].reset_index(drop=True)

# 앙상블 전체의 SHAP 평균 계산
sv_entity_list = []
for model in models:
    exp = shap.TreeExplainer(model)
    sv_e = exp.shap_values(X_entity)
    if isinstance(sv_e, list):
        sv_e = sv_e[1]
    sv_entity_list.append(sv_e)
mean_sv_entity = np.mean(sv_entity_list, axis=0)

# 고장 확률이 가장 높은 시점 선택
peak_idx = df_entity["_prob"].values.argmax()
peak_date = df_entity[cfg.DATE_COL].iloc[peak_idx].date()
peak_prob = df_entity["_prob"].iloc[peak_idx]
print(f"[§9.2] Waterfall Plot — 최고 위험 시점: {peak_date} (예측 확률={peak_prob:.4f})")

# shap.Explanation 객체 생성 (Waterfall 필요)
# 첫 번째 모델 기준 expected_value 사용 (앙상블 평균)
base_vals = np.mean([shap.TreeExplainer(m).expected_value for m in models])
if isinstance(base_vals, np.ndarray):
    base_vals = base_vals[1]

explanation = shap.Explanation(
    values=mean_sv_entity[peak_idx],
    base_values=float(base_vals),
    data=X_entity.iloc[peak_idx].values,
    feature_names=FEATURE_COLS,
)

shap.plots.waterfall(explanation, max_display=15, show=True)

---
## §9.3 시간 기반 해석 (Temporal Interpretation)

고장은 하루아침에 발생하지 않고 점진적으로 열화됩니다.  
고장 전 **`TEMPORAL_WINDOW_DAYS`일** 동안 각 피처의 SHAP 기여도가 어떻게 변화했는지 추적합니다.

- **x축:** 고장 직전 경과일 (D-N → D-1 방향으로 진행)
- **y축:** 해당 피처의 SHAP 값 (양수 = 고장 확률 증가 기여)

In [ ]:
# §9.3 — 고장 전 N일 창 SHAP Trajectory
window_days = cfg.TEMPORAL_WINDOW_DAYS
top_n = cfg.TEMPORAL_TOP_N_FEATS

# 마지막 관측일 기준 최근 N일 필터링
last_date = df_entity[cfg.DATE_COL].max()
start_date = last_date - pd.Timedelta(days=window_days - 1)
df_window = df_entity[df_entity[cfg.DATE_COL] >= start_date].copy().reset_index(drop=True)

print(f"[§9.3] 분석 기간: {start_date.date()} ~ {last_date.date()} ({len(df_window)}일)")

if len(df_window) == 0:
    print("⚠️  분석 기간 내 데이터가 없습니다. TEMPORAL_WINDOW_DAYS를 늘려보세요.")
else:
    X_window = df_window[FEATURE_COLS].reset_index(drop=True)

    # 윈도우 구간 SHAP 계산
    sv_window_list = []
    for model in models:
        exp = shap.TreeExplainer(model)
        sv_w = exp.shap_values(X_window)
        if isinstance(sv_w, list):
            sv_w = sv_w[1]
        sv_window_list.append(sv_w)
    mean_sv_window = np.mean(sv_window_list, axis=0)  # shape: (days, n_feats)

    # 윈도우 구간에서 기여도가 큰 상위 N개 피처 선택
    abs_mean_window = np.abs(mean_sv_window).mean(axis=0)
    top_feat_idx = np.argsort(abs_mean_window)[::-1][:top_n]
    top_feat_names = [FEATURE_COLS[i] for i in top_feat_idx]

    dates = df_window[cfg.DATE_COL].values
    days_to_last = [(last_date - pd.Timestamp(d)).days for d in dates]

    # 시각화
    fig, axes = plt.subplots(top_n, 1, figsize=(12, 3 * top_n), sharex=True)
    if top_n == 1:
        axes = [axes]

    colors = plt.cm.tab10.colors
    for ax, feat_name, feat_idx, color in zip(axes, top_feat_names, top_feat_idx, colors):
        shap_vals = mean_sv_window[:, feat_idx]
        feat_vals = X_window[feat_name].values

        ax2 = ax.twinx()
        ax2.plot(days_to_last, feat_vals, color="lightgray", lw=1.2,
                 linestyle="--", label=f"{feat_name} (원본값)", alpha=0.7)
        ax2.set_ylabel("원본값", color="gray", fontsize=9)
        ax2.tick_params(axis="y", labelcolor="gray")

        ax.bar(days_to_last, shap_vals, color=color, alpha=0.75, width=0.8)
        ax.axhline(0, color="black", lw=0.8, ls="-")
        ax.set_ylabel("SHAP 기여도", fontsize=10)
        ax.set_title(f"{feat_name}", fontsize=11, fontweight="bold")
        ax.grid(axis="y", alpha=0.3)

    axes[-1].set_xlabel("마지막 관측일로부터 경과일 (D-N 방향 →)", fontsize=11)
    axes[-1].invert_xaxis()  # 시간 순서: 왼쪽=과거, 오른쪽=최근

    fig.suptitle(
        f"[§9.3] 시간 기반 SHAP Trajectory\n"
        f"개체: {TARGET_SERIAL}  |  고장 전 {window_days}일 상위 {top_n}개 피처",
        fontsize=13, fontweight="bold", y=1.01
    )
    plt.tight_layout()
    plt.show()
    print(f"\n분석 완료. 상위 {top_n}개 기여 피처: {top_feat_names}")

---
## §9.4 실무 활용 요약

본 해석 분석을 통해 도출할 수 있는 실무적 의미를 정리합니다.

In [ ]:
# §9.4 — 실무 활용 요약 출력
SEP = "=" * 65

print(SEP)
print("  §9.4  실무 활용 효과 요약")
print(SEP)

# 상위 피처 목록 (§9.1 결과 기반)
top5_global = shap_rank_df["feature"].head(5).tolist()
print("\n▶ [전역 해석 - §9.1] 주요 고장 기여 피처 Top 5:")
for rank, feat in enumerate(top5_global, 1):
    score = shap_rank_df.loc[shap_rank_df["feature"] == feat, "mean_abs_shap"].values[0]
    print(f"    {rank}. {feat:<35} (Mean |SHAP| = {score:.5f})")

print(f"\n▶ [국소 해석 - §9.2] 분석 개체: {TARGET_SERIAL}")
print(f"    최고 위험 시점: {peak_date}  (예측 확률 = {peak_prob:.4f})")
print(f"    → Waterfall Plot에서 해당 시점의 개별 피처 기여도 확인 가능")

print(f"\n▶ [시간 해석 - §9.3] 고장 전 {window_days}일 창 분석")
print(f"    추적 피처: {', '.join(top_feat_names)}")
print(f"    → 어느 피처가 언제부터 급격히 기여도가 상승했는지 확인 가능")

print(f"\n▶ 실무 의미:")
print(f"    1. 조기 경보 원인 설명: 특정 SMART 지표가 언제부터 이상 신호를 보냈는지 설명")
print(f"    2. 유지보수 판단 근거: 엔지니어에게 '이 디스크는 A 피처 때문에 경보 발령'이라는 근거 제공")
print(f"    3. 모델 신뢰성 향상: Black-box 모델의 의사결정 과정을 투명하게 공개")
print(SEP)